In [ ]:
import random
import re
from datasets import load_dataset, get_dataset_config_names
from collections import defaultdict

# 🧠 [데이터셋 설명]
# 데이터셋명: nayohan/korean-hate-speech
# 주제: 온라인 댓글 내 혐오 발언 및 편향 분석
# 설명: 이 데이터셋은 한국어 온라인 댓글을 수집하여, 댓글 내용, 성차별적 편향 여부, 편향 유형, 그리고 혐오 발언 여부 등의 레이블링을 포함하고 있습니다.
# 목표: 댓글의 내용과 특정 속성(예: 성별 편향)이 혐오 발언 여부와 어떤 상관관계를 갖는지 탐구합니다.
# 활용 난이도: 초급 (기본 데이터 탐색 및 카운팅 실습)

# =========================================================================
# 🤖 튜터 모드 시작: 튜터의 환영 인사
# =========================================================================
print("✨ 안녕하세요! 튜터 AI가 온디가이드 파이썬 코딩 실습을 시작하겠습니다! ✨")
print("🌟 오늘 우리는 '한국어 혐오 발언 탐지'라는 흥미진진한 주제를 다룰 거예요.")
print("🤔 우리의 목표는 댓글의 내용(comments)만 보는 것이 아니라, 성별 편향성(gender bias)과 혐오 여부(hate) 간의 관계를 통계적으로 탐색해 보는 것입니다.")
print("💡 기억하세요! AI는 단순히 패턴을 학습하는 기계입니다. 데이터에 숨겨진 '의미'를 찾아내는 것이 가장 중요해요!")
print("------------------------------------------------------------------------")


# 📑 데이터 로드 및 설정 (가장 중요!)
DATASET_NAME = "nayohan/korean-hate-speech"
SAMPLE_COUNT = 100  # 속도와 효율성을 위해 100개의 샘플만 사용합니다.

print(f"\n📂 1단계: {DATASET_NAME} 데이터셋 로드 준비...")

# 1. Config 목록 확인 (필수 절차)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0] # 첫 번째 Config를 기본으로 사용
except Exception as e:
    print(f"ℹ️ Config 로드 중 오류가 발생했습니다: {e}. 기본 설정으로 진행합니다.")
    selected_config = None

# 2. 스트리밍 로드 시도 (최신 방식)
dataset = None
try:
    # 스트리밍 모드는 메모리 효율이 좋고 데이터셋이 매우 클 때 좋습니다.
    # split='train'으로 지정하여 학습 데이터에서 테스트 샘플을 가져옵니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("\n✅ 스트리밍 모드(streaming=True)로 데이터셋 로드 성공! 메모리 절약에 최고예요.")
except Exception as e:
    print(f"\n❌ 경고: 스트리밍 모드 로드에 실패했습니다 ({e}). 일반 모드로 전환합니다.")
    # 실패 시, 작은 테스트 세트만 다운로드하여 진행합니다.
    dataset = load_dataset(DATASET_NAME, split='train[:1%]') # 1%만 로드하여 무한 루프 방지
    print("✅ 일반 모드(small subset)로 데이터셋 로드 성공! 작게 테스트합니다.")

# 3. 샘플 데이터 추출 (Constraint 9 & 16 준수)
# 데이터셋 객체 자체를 그대로 사용하지 않고, take()을 사용해 이터레이터로 변환합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"\n🔍 {SAMPLE_COUNT}개 샘플을 샘플링하여 분석을 시작합니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # 이터레이터 내용을 리스트로 변환하여 튜플 형태로 접근합니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

# =========================================================================
# 🔍 2단계: 데이터 탐색 및 전처리 실습
# =========================================================================
print("\n==============================================================")
print("💡 2단계: 데이터 구조 분석 및 Key Feature 추출")
print("==============================================================")

# 데이터 분석에 사용할 빈 카운터를 준비합니다.
# 딕셔너리: { '성별 편향 여부': { '혐오 발언 여부': 카운트 } }
bias_hate_counts = defaultdict(lambda: defaultdict(int))
total_samples = 0
data_points_analyzed = []

print("➡️ [실습 목표] '성별 편향 유무'와 '혐오 발언 여부'의 관계를 카운트해 봅시다.")

# 모든 샘플을 순회하며 필요한 값들만 추출합니다.
for i, sample in enumerate(sample_data_list):
    if i >= SAMPLE_COUNT:
        break # 안전 장치

    try:
        # 🎯 분석에 필요한 Feature 추출
        comments = sample['comments']
        has_bias = sample['contain_gender_bias']
        is_hate = sample['hate']
        
        # 📊 카운팅 로직 실행
        bias_hate_counts[str(has_bias)][str(is_hate)] += 1
        total_samples += 1
        
        # (선택) 데이터 포인트 저장 (나중에 필요할 수 있습니다)
        data_points_analyzed.append({
            'comment': comments,
            'bias': has_bias,
            'hate': is_hate
        })

    except KeyError as e:
        print(f"⚠️ [경고] {e} 키가 누락된 샘플이 있습니다. 다음 샘플로 넘어갑니다.")
        continue
    except Exception as e:
        print(f"⚠️ [오류] {e} 발생. 건너뜁니다.")
        continue

print("\n🎉 데이터 분석 및 카운팅 완료! 총 샘플 분석 개수:", total_samples, "개")


# =========================================================================
# 📈 3단계: 통계적 해석 및 시각적 출력 (튜터의 위트 포인트!)
# =========================================================================
print("\n==============================================================")
print("👑 3단계: 통계적 관계 분석 결과 (Key Insights)")
print("==============================================================")

print("\n🌈 분석 내용: 성별 편향성('contain_gender_bias')에 따라 혐오 발언('hate')이 발생하는 비율을 확인합니다.")
print("------------------------------------------------------------------------------")

# 성별 편향 유무에 따른 혐오 발언 발생 비율을 정리합니다.
print("\n▶️ [결과 1] 성별 편향이 '있다' (True) 인 경우:")
if 'True' in bias_hate_counts:
    true_counts = bias_hate_counts['True']
    total_true = true_counts['True'] + true_counts['False']
    
    hate_count = true_counts['True']
    bias_count = total_true
    
    print(f"    - 총 샘플 (편향 O): {bias_count} 건")
    print(f"    - 💔 혐오 발언 (True) 비율: {hate_count} 건 (비율: {(hate_count / total_true * 100):.2f}%)")
    print(f"    - 💪 혐오 발언이 아닌 비율: {true_counts['False']} 건")
else:
    print("    ⚠️ '성별 편향 O' 샘플이 발견되지 않았습니다.")


print("\n▶️ [결과 2] 성별 편향이 '없다' (False) 인 경우:")
if 'False' in bias_hate_counts:
    false_counts = bias_hate_counts['False']
    total_false = false_counts['True'] + false_counts['False']

    hate_count = false_counts['True']
    bias_count = total_false
    
    print(f"    - 총 샘플 (편향 X): {bias_count} 건")
    print(f"    - 💔 혐오 발언 (True) 비율: {hate_count} 건 (비율: {(hate_count / total_false * 100):.2f}%)")
    print(f"    - 💪 혐오 발언이 아닌 비율: {false_counts['False']} 건")
else:
    print("    ⚠️ '성별 편향 X' 샘플이 발견되지 않았습니다.")


# =========================================================================
# 🎁 4단계: 창의적 실습 예시 - '가장 극단적인 샘플' 탐색 (Troubleshooting)
# =========================================================================
print("\n\n==============================================================")
print("🎁 4단계: 고위험 샘플 Spotlight 분석 (Troubleshooting)")
print("==============================================================")
print("✨ [튜터의 꿀팁] AI가 실수하는 부분을 찾아보는 것이 가장 중요한 공부예요. 가장 강력한 '혐오 발언'이 포함된 샘플을 2개 보여드릴게요.")

# 조건: 혐오 발언(hate='True')이면서, 가장 문장이 길거나, 편향성이 나타난 샘플을 찾습니다.
high_risk_samples = []
for sample in sample_data_list:
    # 혐오 발언 레이블이 'True'인 샘플을 찾습니다.
    if sample['hate'] == 'True':
        high_risk_samples.append(sample)
        if len(high_risk_samples) >= 2:
            break

if high_risk_samples:
    print("\n[🚨 High-Risk Sample 1 (가장 먼저 발견된 혐오 발언)]")
    sample1 = high_risk_samples[0]
    print(f"  - 📝 댓글: {sample1['comments'][:80]}...") # 처음 80자만 출력
    print(f"  - 🌟 성별 편향성 여부: {sample1['contain_gender_bias']}")
    print(f"  - 🗣️ 혐오 발언 여부: {sample1['hate']}")

    print("\n[🚨 High-Risk Sample 2]")
    sample2 = high_risk_samples[1]
    print(f"  - 📝 댓글: {sample2['comments'][:80]}...")
    print(f"  - 🌟 성별 편향성 여부: {sample2['contain_gender_bias']}")
    print(f"  - 🗣️ 혐오 발언 여부: {sample2['hate']}")
    print("\n💡 해설: 이 샘플들을 보면, '편향성'과 '혐오 발언'이 모두 감지되는 경우가 많다는 것을 알 수 있죠? AI 모델을 만들 때는 이 관계를 깊이 있게 학습시켜야 해요!")
else:
    print("⚠️ 분석한 샘플 내에서 혐오 발언('hate': 'True')을 가진 샘플을 찾지 못했습니다.")


print("\n==============================================================")
print("✅ 실습 완료!")
print("축하합니다! 데이터의 구조를 분석하고, 통계적 패턴을 찾아내는 과정을 성공적으로 마쳤습니다.")
print("다음에 만날 때는 이 패턴을 기반으로 '분류 모델'을 직접 만들어 봅시다. 👍")